In [1]:
!apt-get install ncbi-blast+ -y

n
n
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  ncbi-data
The following NEW packages will be installed:
  ncbi-blast+ ncbi-data
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 15.8 MB of archives.
After this operation, 71.8 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-data all 6.1.20170106+dfsg1-9 [3,519 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-blast+ amd64 2.12.0+ds-3build1 [12.3 MB]
Fetched 15.8 MB in 1s (16.7 MB/s)
Selecting previously unselected package ncbi-data.
(Reading database ... 126374 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.d

In [2]:
# 1. Install Biopython
!pip install biopython

# 2. Download Swiss-Prot FASTA
!wget ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/swissprot.gz

# 3. Unzip
!gunzip swissprot.gz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.9 MB/s eta 0:00:00
--2025-09-13 19:40:31--  ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/swissprot.gz
           => ‘swissprot.gz’
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.13, 130.14.250.7, 130.14.250.10, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.13|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /blast/db/FASTA ... done.
==> SIZE swissprot.gz ... 143234298
==> PASV ... done.    ==> RETR swissprot.gz ... done.
Length: 143234298 (137M) (unauthoritative)

swissprot.gz        100%[===================>] 136.60M  60.2MB/s    in 2.3s    

2025-09-13 19:40:34 (60.2 MB/s) - ‘swissprot.gz’ saved [143234298]



In [3]:
from Bio import SeqIO
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter , defaultdict
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch


In [4]:
sequences, labels = [], []

for record in SeqIO.parse("swissprot", "fasta"):
    seq = str(record.seq)
    header = record.description

    # Extract species name inside brackets
    species = re.search(r"\[(.*?)\]", header)
    if species:
        sequences.append(seq)
        labels.append(species.group(1))

print("Total sequences:", len(sequences))
print("Example sequence:", sequences[0])
print("Example label:", labels[0])


Total sequences: 482697
Example sequence: MASKEMFEDTVEERVINEEYKIWKKNTPFLYDLVMTHALQWPSLTVQWLPEVTKPEGKDYALHWLVLGTHTSDEQNHLVVARVHIPNDDAQFDASHCDSDKGEFGGFGSVTGKIECEIKINHEGEVNRARYMPQNPHIIATKTPSSGVLVFDYTKHPAKPDPSGECNPDLRLRGHQKEGYGLSWNSNLSGHLLSASDDHTVCLWDINAGPKEGKIVDAKAVFTGHSAVVEDVAWHLLHESLFGSVADDQKLMMWDTRSNTTSKPSHLVDAHTAEVNCLSFNPYSEFILATGSADKTVALWDLRNLKLKLHTFESHKDEIFQVVHWSPHNETILASSGTDRRLNVWDLSKIGEEQSAEDAEDGPPELLFIHGGHTAKISDFSWNPNEPWVICSVSEDNIMQIWQMAENIYNDEESDVTTSELEGQGS
Example label: Pongo abelii


In [5]:
species = re.search(r"\[(.*?)\]", header)
if species:
    labels.append(species.group(1))
elif "OS=" in header:
    labels.append(header.split("OS=")[1].split("OX=")[0].strip())
else:
    labels.append("Unknown")

In [6]:
print("Total sequences:", len(sequences))
print("Total labels:", len(labels))
print("Unique species:", len(set(labels)))
print("Example:", labels[:10])


Total sequences: 482697
Total labels: 482698
Unique species: 14280
Example: ['Pongo abelii', 'Jannaschia sp. CCS1', 'Homo sapiens', 'Mus musculus', 'Homo sapiens', 'Schizosaccharomyces pombe 972h-', 'Klebsiella pneumoniae 342', 'Walterinnesia aegyptia', 'Bacillus subtilis subsp. subtilis str. 168', 'Cupriavidus necator H16']


In [7]:
counts = Counter(labels)
min_samples = 10  # or 20 depending on dataset size

X = [seq for seq, lbl in zip(sequences, labels) if counts[lbl] >= min_samples]
y = [lbl for lbl in labels if counts[lbl] >= min_samples]

print("After filtering:", len(X), "sequences across", len(set(y)), "species")

After filtering: 454850 sequences across 2697 species


In [8]:
lengths = Counter(len(seq) for seq in X)

print("Unique lengths:", len(lengths))
print("Example mapping (length → count):")
for length, count in list(lengths.items()):  # show first 20
    print(length, ":", count)

Unique lengths: 3305
Example mapping (length → count):
426 : 829
75 : 580
374 : 636
181 : 922
1791 : 6
332 : 832
119 : 942
83 : 725
62 : 595
172 : 920
69 : 452
255 : 981
523 : 329
316 : 948
377 : 742
1304 : 14
576 : 282
158 : 1063
144 : 911
417 : 750
424 : 724
596 : 255
201 : 1208
303 : 871
341 : 1011
458 : 542
428 : 848
307 : 872
408 : 530
2043 : 1
161 : 962
461 : 586
859 : 125
311 : 963
194 : 928
51 : 256
672 : 126
165 : 805
235 : 849
174 : 721
396 : 996
269 : 824
1079 : 37
148 : 924
197 : 978
256 : 942
1050 : 41
327 : 791
666 : 167
364 : 847
894 : 77
231 : 858
875 : 118
376 : 799
209 : 1090
237 : 875
126 : 915
48 : 194
579 : 244
360 : 1064
76 : 564
887 : 103
45 : 208
353 : 883
354 : 910
501 : 492
138 : 899
601 : 295
266 : 788
111 : 667
649 : 156
171 : 769
252 : 1023
911 : 89
317 : 946
195 : 821
55 : 338
309 : 1064
74 : 488
528 : 295
375 : 729
29 : 189
153 : 890
486 : 490
382 : 772
333 : 841
163 : 794
402 : 547
270 : 872
282 : 841
870 : 82
88 : 917
331 : 828
185 : 1233
521 : 315
152 

In [9]:
le = LabelEncoder()
y = torch.tensor(le.fit_transform(labels), dtype=torch.long)
num_classes = len(le.classes_)
print("Classes:", num_classes)


Classes: 14280


In [10]:
# Create amino acid vocabulary
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'  # Standard 20 amino acids
SPECIAL_TOKENS = ['<PAD>', '<UNK>']
vocab = SPECIAL_TOKENS + list(AMINO_ACIDS)
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

def tokenize_sequence(seq, max_len=512):
    """Convert sequence to token indices"""
    tokens = [char_to_idx.get(aa, char_to_idx['<UNK>']) for aa in seq[:max_len]]
    # Pad sequences to max_len
    tokens += [char_to_idx['<PAD>']] * (max_len - len(tokens))
    return tokens

# Tokenize all sequences
max_sequence_length = 512  # Adjust based on your length analysis
X_tokens = [tokenize_sequence(seq, max_sequence_length) for seq in X]
X_tensor = torch.tensor(X_tokens, dtype=torch.long)

In [11]:
# Debug X_tensor and y length mismatch

print("=== DEBUGGING TENSOR MISMATCH ===")

# Check the dimensions of all your variables
print("Current data dimensions:")
print(f"X (sequences): {len(X)}")
print(f"y_labels (string labels): {len(y)}")
print(f"y (encoded tensor): {y.shape}")

# The issue: X_tensor was created from X, but y might be from different filtering
# Let's check if you have X_tensor defined
try:
    print(f"X_tensor: {X_tensor.shape}")
except NameError:
    print("X_tensor is not defined yet")

# Check if the filtering steps were applied consistently
print(f"\nLabel encoder classes: {len(le.classes_)}")
print(f"Number of unique labels in y_labels: {len(set(y))}")

# SOLUTION: Ensure X and y are properly aligned before creating tensors

# Step 1: Verify X and y_labels are aligned
print(f"\n=== VERIFICATION ===")
print(f"len(X): {len(X)}")
print(f"len(y_labels): {len(y)}")

if len(X) != len(y):
    print("ERROR: X and y_labels have different lengths!")

    # Fix by taking the minimum length
    min_len = min(len(X), len(y))
    X = X[:min_len]
    y_labels = y[:min_len]
    print(f"Fixed: Both trimmed to {min_len} samples")

# Step 2: Re-encode labels to ensure consistency
le = LabelEncoder()
y = torch.tensor(le.fit_transform(y_labels), dtype=torch.long)
num_classes = len(le.classes_)

print(f"After re-encoding:")
print(f"y shape: {y.shape}")
print(f"num_classes: {num_classes}")

# Step 3: Tokenize sequences (this should match y length)
# Create amino acid vocabulary
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'  # Standard 20 amino acids
SPECIAL_TOKENS = ['<PAD>', '<UNK>']
vocab = SPECIAL_TOKENS + list(AMINO_ACIDS)
char_to_idx = {char: idx for idx, char in enumerate(vocab)}

def tokenize_sequence(seq, max_len=512):
    """Convert sequence to token indices"""
    tokens = [char_to_idx.get(aa, char_to_idx['<UNK>']) for aa in seq[:max_len]]
    # Pad sequences to max_len
    tokens += [char_to_idx['<PAD>']] * (max_len - len(tokens))
    return tokens

# Choose max length based on your analysis
max_sequence_length = 512  # Start with this for faster training

print(f"\nTokenizing sequences with max_length={max_sequence_length}...")
X_tokens = [tokenize_sequence(seq, max_sequence_length) for seq in X]
X_tensor = torch.tensor(X_tokens, dtype=torch.long)

print(f"X_tensor shape: {X_tensor.shape}")
print(f"y shape: {y.shape}")

# Step 4: Final verification before train_test_split
assert X_tensor.shape[0] == y.shape[0], f"Shape mismatch: X_tensor {X_tensor.shape[0]} vs y {y.shape[0]}"
print(f"✓ Shapes are aligned: {X_tensor.shape[0]} samples")

# Step 5: Now the train_test_split should work
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_tensor, y, test_size=0.2, random_state=42
)

print(f"\n=== TRAIN/TEST SPLIT SUCCESSFUL ===")
print(f"Training samples: {len(X_train)} (shape: {X_train.shape})")
print(f"Validation samples: {len(X_val)} (shape: {X_val.shape})")
print(f"Training labels: {len(y_train)} (shape: {y_train.shape})")
print(f"Validation labels: {len(y_val)} (shape: {y_val.shape})")

# Quick sanity check
print(f"\nSample data:")
print(f"First sequence tokens (first 10): {X_train[0][:10].tolist()}")
print(f"First few labels: {y_train[:5].tolist()}")
print(f"Corresponding species: {[le.inverse_transform([label.item()])[0] for label in y_train[:5]]}")

=== DEBUGGING TENSOR MISMATCH ===
Current data dimensions:
X (sequences): 454850
y_labels (string labels): 482698
y (encoded tensor): torch.Size([482698])
X_tensor: torch.Size([454850, 512])

Label encoder classes: 14280
Number of unique labels in y_labels: 482698

=== VERIFICATION ===
len(X): 454850
len(y_labels): 482698
ERROR: X and y_labels have different lengths!
Fixed: Both trimmed to 454850 samples
After re-encoding:
y shape: torch.Size([454850])
num_classes: 13904

Tokenizing sequences with max_length=512...
X_tensor shape: torch.Size([454850, 512])
y shape: torch.Size([454850])
✓ Shapes are aligned: 454850 samples

=== TRAIN/TEST SPLIT SUCCESSFUL ===
Training samples: 363880 (shape: torch.Size([363880, 512]))
Validation samples: 90970 (shape: torch.Size([90970, 512]))
Training labels: 363880 (shape: torch.Size([363880]))
Validation labels: 90970 (shape: torch.Size([90970]))

Sample data:
First sequence tokens (first 10): [12, 18, 17, 7, 14, 6, 6, 6, 3, 19]
First few labels: [49

In [12]:
import torch.nn as nn

class ProteinCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=5, padding=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.embedding(x)  # (batch, seq_len, embed_dim)
        x = x.transpose(1, 2)  # (batch, embed_dim, seq_len)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)  # (batch, 256)
        x = self.dropout(x)
        return self.fc(x)

In [13]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Initialize model
vocab_size = len(vocab)
model = ProteinCNN(vocab_size=vocab_size, embed_dim=128, num_classes=num_classes)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def train_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # move both to GPU

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # move both to GPU
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    accuracy = 100 * correct / total
    return total_loss / len(val_loader), accuracy

# Training loop
epochs = 50
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validate(model, val_loader, criterion)

    print(f'Epoch {epoch}: Train Loss: {train_loss:.4f}, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
